# SI 618 WN Project Part III

## Team Members

#### Carlos Figueredo (carlosfc), James Zhu (jazhu)


In [105]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import gzip

In [106]:
def read_births():
    births = pd.DataFrame()
    for y in range(2011, 2014):
        da = pd.read_csv("%4d.txt" % y, delimiter="\t", dtype={"County Code": object})
        da = da[["County", "County Code", "Births"]]
        da["year"] = y
        births = pd.concat([births, da])
    births = births.dropna()
    births = births.rename({'County Code':'FIPS'}, axis=1)
    unidentified_mask = births['County'].str.contains('Unidentified')
    births = births[~ unidentified_mask]
    births = births.groupby(['FIPS']).agg(Births=('Births', 'mean'),
                                 StdBirths=('Births', 'std'))
    return births

births = read_births()
births.head()

,Births,StdBirths
FIPS,,
01003,2142.000000,28.618176
01015,1353.333333,61.744096
01055,1167.666667,17.616280
01073,8861.000000,199.276190
01081,1724.000000,167.654406


In [107]:
def read_rucc():
    rucc = pd.read_excel("ruralurbancodes2013.xls", sheet_name=None) # Requires xlrd. Run pip install xlrd
    rucc = rucc["Rural-urban Continuum Code 2013"] # Get first sheet
    rucc["FIPS"] = ["%05d" % x for x in rucc.FIPS] # FIPS to object dtype
    rucc['RUCC_Category'] = rucc['RUCC_2013'].apply(lambda x: "Metro" if x <= 3 else "Nonmetro")
    rucc = rucc.dropna()
    return rucc

# rucc = read_rucc()
# rucc.head()

In [108]:
def read_demog():
    grid = [1, 5, 7, 9, 12, 14, 15, 16, 17, 19, 27]
    ranges = [(grid[i]-1, grid[i+1]-1) for i in range(len(grid)-1)]
    with gzip.open("2016ages.txt.gz") as io:
        demog = pd.read_fwf(io, colspecs=ranges, header=None)
    demog.columns = ["Year", "State", "StateFIPS", "CountyFIPS", "Registry",
                    "Race", "Origin", "Sex", "Age", "Population"]
    demog["FIPS"] = ["%02d%03d" % (x, y) for (x, y) in zip(demog.StateFIPS, demog.CountyFIPS)]
    demog = demog[["FIPS", "Race", "Age", "Population"]]
    demog["Race"] = demog["Race"].replace([1, 2, 3, 4], ["W", "B", "N", "A"]) # White/Black/Native/Asian
    # Age group labels (For Reference)
    age_groups = ["0", "1-4", "5-9", "10-14", "15-19", "20-24", "25-29", "30-34", "35-39",
                "40-44", "45-49", "50-54", "55-59", "60-64", "65-69", "70-74", "75-79",
                "80-84", "85-89", "90+"]
    return demog

def pivot_demog(demog):
    demog = demog.pivot_table(index="FIPS", columns=["Race", "Age"], values="Population", aggfunc="sum")
    na = demog.columns.tolist()
    demog.columns = ["%s:%d" % tuple(x) for x in na]
    demog = demog.fillna(0)
    return demog

# demog = read_demog()
# demog = pivot_demog(demog)
# demog.head()


In [109]:
def read_data(drop_FIPS=True):
    births = read_births()
    demog = read_demog()
    demog = pivot_demog(demog)
    demog_sum = demog.sum(axis=1).to_frame()
    demog_sum.columns = ["Population"]
    births_demog = pd.merge(births, demog_sum, on="FIPS")
    births_demog["Births"] = births_demog["Births"] / births_demog["Population"]
    births_demog["StdBirths"] = births_demog["StdBirths"] / births_demog["Population"]
    births_demog = births_demog.drop(columns=["Population"])
    norm_demog = demog.div(demog.sum(axis=1), axis=0)
    df = pd.merge(births_demog, norm_demog, on="FIPS", how="right")
    rucc = read_rucc()[["FIPS", "RUCC_Category"]]
    rucc["Metro"] = rucc["RUCC_Category"].apply(lambda x: 1 if x == "Metro" else 0)
    rucc = rucc.drop(columns=["RUCC_Category"])
    df = pd.merge(df, rucc, on="FIPS")
    if drop_FIPS:
        df = df.drop(columns=["FIPS"])
    return df

df = read_data()

In [110]:
df.isna().sum()

Births       2617
StdBirths    2617
A:0             0
A:1             0
A:2             0
             ... 
W:16            0
W:17            0
W:18            0
W:19            0
Metro           0
Length: 83, dtype: int64

## Imputation of Births

In [111]:
def impute(row, column):
    if pd.isna(row[column]):
        return 'Unknown'
    elif row[column] > df[column].median():
        return 'High'
    else:
        return 'Low'

df['Births'] = df.apply(lambda row: impute(row, 'Births'), axis=1)
df['StdBirths'] = df.apply(lambda row: impute(row, 'StdBirths'), axis=1)

In [112]:
df.isna().sum().sum()

0

## Clasification

In [113]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [114]:
X = df.drop(columns=["Metro"])
y = df["Metro"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [115]:
onehot_encoder = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(), ['Births', 'StdBirths']),
    ],
    remainder='passthrough'  # Keep other columns as is
)
model = Pipeline([
    ('preprocessor', onehot_encoder),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

In [116]:
model.fit(X_train, y_train)

# Predict on the test data
y_pred = model.predict(X_test)
# Print classification report
print(classification_report(y_test, y_pred))
# Print confusion matrix
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.83      0.94      0.88       406
           1       0.86      0.65      0.74       223

    accuracy                           0.84       629
   macro avg       0.85      0.79      0.81       629
weighted avg       0.84      0.84      0.83       629

[[383  23]
 [ 79 144]]
